# 09_extraccion_riesgos.ipynb

**Actividad de Grado MIA UC**  
**Sistema Inteligente de Vigilancia de Riesgos Documentales basado en RAG y LLMs**

Este notebook implementa la primera versión del módulo de **extracción estructurada de riesgos**.

## Objetivo
Analizar chunks documentales y extraer riesgos en formato estructurado usando LLMs.

## Arquitectura

```text
chunks_recursive.parquet
        ↓
Selección de chunks candidatos
        ↓
LLM con prompt controlado
        ↓
JSON estructurado
        ↓
Tabla de riesgos
```

## Entradas
- `chunks_recursive.parquet`

## Salidas
- `riesgos_extraidos.csv`
- `riesgos_extraidos.xlsx`
- `riesgos_por_categoria.csv`
- `riesgos_por_documento.csv`
- `riesgos_extraccion_log.csv`


In [ ]:
# =====================================================
# 09_extraccion_riesgos.ipynb
# Actividad de Grado MIA UC
# Patricia Patiño
# =====================================================

!pip -q install pandas pyarrow openpyxl openai

## 1. Importar librerías

In [ ]:
import os
import json
import re
import time
import pandas as pd
from pathlib import Path
from getpass import getpass
from google.colab import files

pd.set_option('display.max_colwidth', 160)

print('Librerías cargadas correctamente')

## 2. Cargar chunks

Sube el archivo:

`chunks_recursive.parquet`

In [ ]:
print('Sube chunks_recursive.parquet')
uploaded = files.upload()
print('\nArchivos cargados:', list(uploaded.keys()))

## 3. Leer y validar chunks

In [ ]:
chunks_path = Path('chunks_recursive.parquet')

if not chunks_path.exists():
    raise FileNotFoundError('No se encontró chunks_recursive.parquet')

chunks_df = pd.read_parquet(chunks_path).reset_index(drop=True)

required_cols = {'doc_id', 'filename', 'tipo_documento', 'page', 'chunk_id', 'chunk_text'}
missing = required_cols - set(chunks_df.columns)

if missing:
    raise ValueError(f'Faltan columnas: {missing}')

print('Chunks cargados:', len(chunks_df))
display(chunks_df.head(3))
display(chunks_df['tipo_documento'].value_counts().reset_index())

## 4. Seleccionar chunks candidatos

Para una primera versión, no se procesan todos los chunks. Se priorizan fragmentos con palabras asociadas a riesgo documental.

In [ ]:
risk_keywords = [
    'riesgo', 'incumplimiento', 'retraso', 'demora', 'vencido', 'vencimiento',
    'no cumple', 'no cumplimiento', 'pendiente', 'atraso', 'desviación', 'desviacion',
    'ans', 'sla', 'compromiso', 'hallazgo', 'observación', 'observacion',
    'calidad', 'despliegue', 'cierre', 'entrega', 'entregable', 'contrato',
    'proveedor', 'interventoría', 'interventoria', 'plan de cierre'
]

pattern = '|'.join([re.escape(k) for k in risk_keywords])

candidate_chunks = chunks_df[
    chunks_df['chunk_text'].str.lower().str.contains(pattern, na=False)
].copy()

print('Chunks candidatos por palabras de riesgo:', len(candidate_chunks))

# Para controlar costo y validar el método, procesamos máximo 60 chunks candidatos.
MAX_CHUNKS_TO_PROCESS = 60

sample_chunks = (
    candidate_chunks
    .sort_values(['tipo_documento', 'doc_id', 'page', 'chunk_id'])
    .head(MAX_CHUNKS_TO_PROCESS)
    .copy()
)

print('Chunks seleccionados para extracción:', len(sample_chunks))
display(sample_chunks[['doc_id', 'filename', 'tipo_documento', 'page', 'chunk_id', 'chunk_size_words']].head(10))

## 5. Configurar API de OpenAI

Se usa un LLM para extraer riesgos en formato JSON controlado.

In [ ]:
api_key = getpass('Ingrese OPENAI_API_KEY: ')
os.environ['OPENAI_API_KEY'] = api_key

from openai import OpenAI
client = OpenAI()

LLM_MODEL = 'gpt-4o-mini'

print('Modelo LLM:', LLM_MODEL)

## 6. Definir esquema de riesgos

El LLM debe devolver una lista JSON. Cada riesgo debe seguir este esquema:

```json
{
  "risk_name": "...",
  "risk_category": "Contractual | Cronograma | Calidad | Operación | Despliegues | Gobierno del Proyecto | Financiero | Seguridad | Otro",
  "severity": "Baja | Media | Alta",
  "probability": "Baja | Media | Alta",
  "risk_description": "...",
  "evidence": "...",
  "recommended_action": "..."
}
```

In [ ]:
VALID_CATEGORIES = [
    'Contractual', 'Cronograma', 'Calidad', 'Operación', 'Despliegues',
    'Gobierno del Proyecto', 'Financiero', 'Seguridad', 'Otro'
]

VALID_LEVELS = ['Baja', 'Media', 'Alta']

def build_risk_extraction_prompt(row):
    text = str(row['chunk_text'])[:4500]

    prompt = f"""
Eres un experto en interventoría, aseguramiento técnico y gestión de riesgos en proyectos de tecnología.

Analiza el siguiente fragmento documental y extrae únicamente riesgos explícitos o razonablemente inferibles a partir del texto.

No inventes riesgos. Si no hay riesgo, devuelve una lista vacía: []

Categorías válidas:
- Contractual
- Cronograma
- Calidad
- Operación
- Despliegues
- Gobierno del Proyecto
- Financiero
- Seguridad
- Otro

Niveles válidos para severity y probability:
- Baja
- Media
- Alta

Devuelve SOLO JSON válido, sin markdown, sin explicación adicional.

Formato esperado:
[
  {{
    "risk_name": "nombre corto del riesgo",
    "risk_category": "una categoría válida",
    "severity": "Baja|Media|Alta",
    "probability": "Baja|Media|Alta",
    "risk_description": "descripción breve del riesgo",
    "evidence": "frase textual o resumen fiel del fragmento que soporta el riesgo",
    "recommended_action": "acción sugerida para mitigar o hacer seguimiento"
  }}
]

Metadatos del fragmento:
- doc_id: {row['doc_id']}
- archivo: {row['filename']}
- tipo_documento: {row['tipo_documento']}
- página: {row['page']}
- chunk_id: {row['chunk_id']}

Fragmento documental:
{text}
"""
    return prompt.strip()

## 7. Funciones de extracción y parseo JSON

In [ ]:
def call_llm_for_risks(prompt, model=LLM_MODEL):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                'role': 'system',
                'content': 'Extrae riesgos documentales en JSON válido. No inventes información.'
            },
            {
                'role': 'user',
                'content': prompt
            }
        ],
        temperature=0.0
    )

    return response.choices[0].message.content


def parse_json_list(raw_text):
    """Intenta convertir la respuesta del LLM en una lista JSON."""
    if raw_text is None:
        return []

    text = raw_text.strip()
    text = text.replace('```json', '').replace('```', '').strip()

    try:
        parsed = json.loads(text)
        if isinstance(parsed, list):
            return parsed
        if isinstance(parsed, dict):
            return [parsed]
        return []
    except Exception:
        # Recuperación simple: buscar lista JSON dentro del texto
        match = re.search(r'\[.*\]', text, flags=re.DOTALL)
        if match:
            try:
                parsed = json.loads(match.group(0))
                if isinstance(parsed, list):
                    return parsed
            except Exception:
                return []
        return []


def normalize_category(value):
    if value in VALID_CATEGORIES:
        return value
    return 'Otro'


def normalize_level(value):
    if value in VALID_LEVELS:
        return value
    return 'Media'

## 8. Ejecutar extracción de riesgos

In [ ]:
risk_records = []
log_records = []

for i, row in sample_chunks.iterrows():
    print(f"Procesando {row['chunk_id']} - {row['filename']}")

    prompt = build_risk_extraction_prompt(row)

    try:
        raw_response = call_llm_for_risks(prompt)
        parsed_risks = parse_json_list(raw_response)

        log_records.append({
            'doc_id': row['doc_id'],
            'filename': row['filename'],
            'chunk_id': row['chunk_id'],
            'status': 'ok',
            'num_risks': len(parsed_risks),
            'raw_response': raw_response
        })

        for risk in parsed_risks:
            risk_records.append({
                'source_doc_id': row['doc_id'],
                'source_filename': row['filename'],
                'source_tipo_documento': row['tipo_documento'],
                'source_page': row['page'],
                'source_chunk_id': row['chunk_id'],
                'risk_name': str(risk.get('risk_name', '')).strip(),
                'risk_category': normalize_category(risk.get('risk_category', 'Otro')),
                'severity': normalize_level(risk.get('severity', 'Media')),
                'probability': normalize_level(risk.get('probability', 'Media')),
                'risk_description': str(risk.get('risk_description', '')).strip(),
                'evidence': str(risk.get('evidence', '')).strip(),
                'recommended_action': str(risk.get('recommended_action', '')).strip(),
                'llm_model': LLM_MODEL
            })

    except Exception as e:
        log_records.append({
            'doc_id': row['doc_id'],
            'filename': row['filename'],
            'chunk_id': row['chunk_id'],
            'status': 'error',
            'num_risks': 0,
            'raw_response': str(e)
        })

    time.sleep(0.3)

risks_df = pd.DataFrame(risk_records)
log_df = pd.DataFrame(log_records)

print('\nExtracción finalizada')
print('Riesgos extraídos:', len(risks_df))
print('Chunks procesados:', len(log_df))

display(risks_df.head())
display(log_df[['chunk_id', 'status', 'num_risks']].head())

## 9. Limpieza y asignación de identificadores

In [ ]:
if risks_df.empty:
    print('No se extrajeron riesgos. Revisa muestra, prompt o respuestas del LLM.')
else:
    risks_df = risks_df.drop_duplicates(
        subset=['source_chunk_id', 'risk_name', 'risk_category', 'evidence']
    ).reset_index(drop=True)

    risks_df.insert(0, 'risk_id', [f'RISK_{i:04d}' for i in range(1, len(risks_df) + 1)])

    # Score simple para priorización inicial
    severity_score_map = {'Baja': 1, 'Media': 2, 'Alta': 3}
    probability_score_map = {'Baja': 1, 'Media': 2, 'Alta': 3}

    risks_df['severity_score'] = risks_df['severity'].map(severity_score_map).fillna(2)
    risks_df['probability_score'] = risks_df['probability'].map(probability_score_map).fillna(2)
    risks_df['risk_score'] = risks_df['severity_score'] * risks_df['probability_score']

    display(risks_df.head(10))
    print('Riesgos únicos:', len(risks_df))

## 10. Análisis descriptivo de riesgos extraídos

In [ ]:
if not risks_df.empty:
    riesgos_por_categoria = (
        risks_df.groupby('risk_category')
        .size()
        .reset_index(name='cantidad_riesgos')
        .sort_values('cantidad_riesgos', ascending=False)
    )

    riesgos_por_documento = (
        risks_df.groupby(['source_doc_id', 'source_filename'])
        .agg(
            cantidad_riesgos=('risk_id', 'count'),
            max_risk_score=('risk_score', 'max'),
            avg_risk_score=('risk_score', 'mean')
        )
        .reset_index()
        .sort_values(['cantidad_riesgos', 'max_risk_score'], ascending=False)
    )

    riesgos_por_severidad = (
        risks_df.groupby(['severity', 'probability'])
        .size()
        .reset_index(name='cantidad_riesgos')
        .sort_values('cantidad_riesgos', ascending=False)
    )

    display(riesgos_por_categoria)
    display(riesgos_por_documento.head(10))
    display(riesgos_por_severidad)
else:
    riesgos_por_categoria = pd.DataFrame()
    riesgos_por_documento = pd.DataFrame()
    riesgos_por_severidad = pd.DataFrame()

## 11. Plantilla de evaluación manual de riesgos

Esta tabla permite revisar si el riesgo extraído es correcto.

Columnas a diligenciar manualmente:
- `riesgo_valido_manual`: 1 si el riesgo es válido, 0 si no.
- `categoria_correcta_manual`: 1 si la categoría es correcta, 0 si no.
- `comentario_evaluador`: observaciones.

In [ ]:
if not risks_df.empty:
    risks_evaluation_template = risks_df.copy()
    risks_evaluation_template['riesgo_valido_manual'] = ''
    risks_evaluation_template['categoria_correcta_manual'] = ''
    risks_evaluation_template['comentario_evaluador'] = ''

    display(risks_evaluation_template.head())
else:
    risks_evaluation_template = pd.DataFrame()

## 12. Guardar artefactos

In [ ]:
if not risks_df.empty:
    risks_df.to_csv('riesgos_extraidos.csv', index=False, encoding='utf-8-sig')
    risks_df.to_excel('riesgos_extraidos.xlsx', index=False)
    risks_evaluation_template.to_excel('riesgos_evaluation_template.xlsx', index=False)
    riesgos_por_categoria.to_csv('riesgos_por_categoria.csv', index=False, encoding='utf-8-sig')
    riesgos_por_documento.to_csv('riesgos_por_documento.csv', index=False, encoding='utf-8-sig')
    riesgos_por_severidad.to_csv('riesgos_por_severidad.csv', index=False, encoding='utf-8-sig')

log_df.to_csv('riesgos_extraccion_log.csv', index=False, encoding='utf-8-sig')

print('Archivos generados:')
if not risks_df.empty:
    print('- riesgos_extraidos.csv')
    print('- riesgos_extraidos.xlsx')
    print('- riesgos_evaluation_template.xlsx')
    print('- riesgos_por_categoria.csv')
    print('- riesgos_por_documento.csv')
    print('- riesgos_por_severidad.csv')
print('- riesgos_extraccion_log.csv')

## 13. Descargar artefactos

In [ ]:
if not risks_df.empty:
    files.download('riesgos_extraidos.csv')
    files.download('riesgos_extraidos.xlsx')
    files.download('riesgos_evaluation_template.xlsx')
    files.download('riesgos_por_categoria.csv')
    files.download('riesgos_por_documento.csv')
    files.download('riesgos_por_severidad.csv')

files.download('riesgos_extraccion_log.csv')